# 07g — BCL POI 2008 production-Beijing normalization

Normalize the validated BCL 2008 FileGDB into a compact Parquet subset for the **same 100 km Beijing study-region policy used by production Home/Office inference**.

This is not an administrative Beijing boundary and it is not a semantic WORK/OFFICE stage.

Pipeline:

1. validate Stage-07f handoff;
2. GDAL bbox pushdown into `POI2008All.gdb` / `POI2008CN`;
3. chunked exact Haversine <=100 km filter;
4. preserve raw `PNAME`, `X`, `Y` separately from geometry lon/lat;
5. write ZSTD Parquet to the persistent Modal Volume;
6. compute geometry/raw-coordinate/duplicate/hash QC;
7. no semantic label creation.


In [ ]:
from pathlib import Path
import importlib.util, json, os, subprocess, sys

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='eda/07g-bcl-beijing-normalization'
REPO_DIR=Path('/root/geolife')

if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())


In [ ]:
import pandas as pd

spec=importlib.util.spec_from_file_location('s07g',REPO_DIR/'analysis'/'07g_bcl_beijing_normalization.py')
s07g=importlib.util.module_from_spec(spec); sys.modules[spec.name]=s07g; spec.loader.exec_module(s07g)
print(s07g.synthetic_self_check())

V=Path('/mnt/geolife-data')
if not V.exists(): raise FileNotFoundError('Attach the GeoLife Modal Volume at /mnt/geolife-data')
C07F=V/'cache'/'07f_bcl_poi_2008_acquisition'
C=V/'cache'/'07g_bcl_beijing_normalization'; C.mkdir(parents=True,exist_ok=True)
OUT_DIR=V/'external'/'bcl_poi_2008'/'normalized'; OUT_DIR.mkdir(parents=True,exist_ok=True)

acq_manifest=json.loads((C07F/'acquisition_manifest.json').read_text(encoding='utf-8'))
worker07f=json.loads((C07F/'worker_report.json').read_text(encoding='utf-8'))
handoff=s07g.validate_07f_handoff(acq_manifest,worker07f)
display(handoff)

region=s07g.production_study_region()
bbox=s07g.study_region_bbox(region)
print('study region:',region)
print('coarse bbox:',bbox)

source_md5=str(handoff.iloc[0]['source_md5'])
gdb_path=str(handoff.iloc[0]['container_path'])
fingerprint=s07g.normalization_fingerprint(source_md5=source_md5,container_path=gdb_path,region=region)
print('normalization fingerprint:',fingerprint)


## Modal worker

The worker reads the existing FileGDB from the persistent Volume. It does not redownload BCL.

Set `GEOLIFE_MODAL_VOLUME_NAME` only if the Modal Volume object is not named `geolife-data`.


In [ ]:
import modal

volume_name=os.environ.get('GEOLIFE_MODAL_VOLUME_NAME','geolife-data')
data_volume=modal.Volume.from_name(volume_name,create_if_missing=False)
app=modal.App('geolife-stage07g-bcl-beijing-normalization')
worker_image=(modal.Image.debian_slim(python_version=f'{sys.version_info.major}.{sys.version_info.minor}')
    .apt_install('gdal-bin')
    .pip_install('numpy','pandas','pyarrow','duckdb'))

@app.function(image=worker_image,volumes={'/mnt/geolife-data':data_volume},timeout=3600,serialized=True)
def normalize_bcl_beijing(payload: dict) -> dict:
    from hashlib import sha256
    from pathlib import Path
    import json, math, re, subprocess
    import duckdb
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq

    root=Path('/mnt/geolife-data')
    cache=root/'cache'/'07g_bcl_beijing_normalization'; cache.mkdir(parents=True,exist_ok=True)
    out_dir=root/'external'/'bcl_poi_2008'/'normalized'; out_dir.mkdir(parents=True,exist_ok=True)
    output=out_dir/'bcl_poi_2008_beijing_100km_v1.parquet'
    result_path=cache/'normalization_worker_result.json'

    def file_sha256(path):
        h=sha256()
        with Path(path).open('rb') as f:
            for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
        return h.hexdigest()

    if output.exists() and result_path.exists():
        previous=json.loads(result_path.read_text(encoding='utf-8'))
        if previous.get('normalization_fingerprint')==payload['fingerprint']:
            current_hash=file_sha256(output)
            if current_hash==previous.get('parquet_sha256'):
                previous['cached']=True
                return previous

    gdb=Path(payload['gdb_path'])
    if not gdb.exists() or not gdb.is_dir():
        return {'ok':False,'error':f'validated FileGDB not found: {gdb}'}

    def run(cmd,timeout=2400):
        try:
            cp=subprocess.run(cmd,capture_output=True,text=True,timeout=timeout)
            return {'returncode':cp.returncode,'stdout':cp.stdout[:500000],'stderr':cp.stderr[:100000]}
        except Exception as exc:
            return {'returncode':-1,'stdout':'','stderr':f'{type(exc).__name__}: {exc}'}

    layer_info=run(['ogrinfo','-ro','-so',str(gdb),payload['layer']],timeout=300)
    if layer_info['returncode']!=0:
        return {'ok':False,'error':'ogrinfo layer validation failed','detail':layer_info}
    info_text=layer_info['stdout']
    if payload['layer'] not in info_text or 'Geometry: Point' not in info_text or 'Feature Count: 6039158' not in info_text or '4326' not in info_text:
        return {'ok':False,'error':'validated layer/geometry/CRS contract changed','ogrinfo':info_text[:20000]}

    staging=Path('/tmp/07g_bcl_beijing_bbox.csv')
    if staging.exists(): staging.unlink()
    bbox=payload['bbox']
    ogr_cmd=[
        'ogr2ogr','-f','CSV','-overwrite',
        '-spat',*(f'{value:.12f}' for value in bbox),
        '-select','PNAME,X,Y','-lco','GEOMETRY=AS_WKT',
        str(staging),str(gdb),payload['layer'],
    ]
    extract=run(ogr_cmd,timeout=2400)
    if extract['returncode']!=0 or not staging.exists():
        return {'ok':False,'error':'ogr2ogr bbox extraction failed','detail':extract,'command':ogr_cmd}

    columns=['poi_id','pname_raw','geom_lon','geom_lat','raw_x','raw_y','distance_to_beijing_km','raw_geom_delta_m','source_year','source_layer','source_crs']
    schema=pa.schema([
        ('poi_id',pa.int64()),('pname_raw',pa.string()),('geom_lon',pa.float64()),('geom_lat',pa.float64()),
        ('raw_x',pa.float64()),('raw_y',pa.float64()),('distance_to_beijing_km',pa.float64()),('raw_geom_delta_m',pa.float64()),
        ('source_year',pa.int16()),('source_layer',pa.string()),('source_crs',pa.string()),
    ])
    point_re=r'^\\s*POINT(?:\\s+(?:Z|M|ZM))?\\s*\\(\\s*([-+0-9.eE]+)\\s+([-+0-9.eE]+)'
    earth=6371008.8
    center_lat=float(payload['region']['latitude']); center_lon=float(payload['region']['longitude']); radius_m=float(payload['region']['radius_km'])*1000.0

    def haversine(lat1,lon1,lat2,lon2):
        a1=np.radians(np.asarray(lat1,dtype=float)); o1=np.radians(np.asarray(lon1,dtype=float))
        a2=np.radians(np.asarray(lat2,dtype=float)); o2=np.radians(np.asarray(lon2,dtype=float))
        da=a2-a1; do=o2-o1
        h=np.sin(da/2.0)**2+np.cos(a1)*np.cos(a2)*np.sin(do/2.0)**2
        return 2.0*earth*np.arcsin(np.sqrt(np.clip(h,0.0,1.0)))

    tmp_output=output.with_suffix('.parquet.part')
    if tmp_output.exists(): tmp_output.unlink()
    writer=pq.ParquetWriter(tmp_output,schema,compression='zstd',use_dictionary=True)
    bbox_rows=0; exact_rows=0; next_id=0; invalid_geometry=0; blank_names=0; raw_xy_valid=0; raw_xy_within_1m=0
    min_lon=math.inf; max_lon=-math.inf; min_lat=math.inf; max_lat=-math.inf; max_distance=0.0
    delta_parts=[]
    try:
        for chunk in pd.read_csv(staging,chunksize=int(payload['chunksize'])):
            bbox_rows += len(chunk)
            required={'WKT','PNAME','X','Y'}
            missing=required.difference(chunk.columns)
            if missing: raise ValueError(f'GDAL CSV missing columns: {sorted(missing)}; got={list(chunk.columns)}')

            geom=chunk['WKT'].astype('string').str.extract(point_re)
            geom_lon=pd.to_numeric(geom[0],errors='coerce'); geom_lat=pd.to_numeric(geom[1],errors='coerce')
            raw_x=pd.to_numeric(chunk['X'],errors='coerce'); raw_y=pd.to_numeric(chunk['Y'],errors='coerce')
            valid=geom_lon.between(-180,180)&geom_lat.between(-90,90)
            invalid_geometry += int((~valid).sum())
            distance=np.full(len(chunk),np.nan,dtype=float)
            if valid.any(): distance[valid.to_numpy()]=haversine(geom_lat[valid],geom_lon[valid],center_lat,center_lon)
            exact=valid & pd.Series(distance<=radius_m+1e-6,index=chunk.index)
            if not exact.any(): continue

            idx=exact.to_numpy(); n=int(idx.sum()); exact_rows += n
            glon=geom_lon[exact].to_numpy(float); glat=geom_lat[exact].to_numpy(float)
            rx=raw_x[exact].to_numpy(float); ry=raw_y[exact].to_numpy(float); dist=distance[idx]
            names=chunk.loc[exact,'PNAME'].astype('string')
            raw_valid=np.isfinite(rx)&np.isfinite(ry)&(rx>=-180)&(rx<=180)&(ry>=-90)&(ry<=90)
            delta=np.full(n,np.nan,dtype=float)
            if raw_valid.any(): delta[raw_valid]=haversine(glat[raw_valid],glon[raw_valid],ry[raw_valid],rx[raw_valid])
            valid_delta=delta[np.isfinite(delta)]
            if len(valid_delta): delta_parts.append(valid_delta)

            blank_names += int((names.isna()|names.str.strip().eq('')).sum())
            raw_xy_valid += int(raw_valid.sum()); raw_xy_within_1m += int((valid_delta<=1.0).sum())
            min_lon=min(min_lon,float(np.min(glon))); max_lon=max(max_lon,float(np.max(glon)))
            min_lat=min(min_lat,float(np.min(glat))); max_lat=max(max_lat,float(np.max(glat))); max_distance=max(max_distance,float(np.max(dist/1000.0)))

            table=pa.table({
                'poi_id':pa.array(np.arange(next_id,next_id+n,dtype=np.int64),type=pa.int64()),
                'pname_raw':pa.array(names.where(names.notna(),None).tolist(),type=pa.string()),
                'geom_lon':pa.array(glon,type=pa.float64()),'geom_lat':pa.array(glat,type=pa.float64()),
                'raw_x':pa.array(rx,type=pa.float64()),'raw_y':pa.array(ry,type=pa.float64()),
                'distance_to_beijing_km':pa.array(dist/1000.0,type=pa.float64()),'raw_geom_delta_m':pa.array(delta,type=pa.float64()),
                'source_year':pa.array(np.full(n,2008,dtype=np.int16),type=pa.int16()),
                'source_layer':pa.array([payload['layer']]*n,type=pa.string()),'source_crs':pa.array(['EPSG:4326']*n,type=pa.string()),
            },schema=schema)
            writer.write_table(table,row_group_size=min(n,100000)); next_id += n
    finally:
        writer.close()

    if exact_rows==0:
        if tmp_output.exists(): tmp_output.unlink()
        return {'ok':False,'error':'exact 100 km filter produced zero POIs','bbox_rows':bbox_rows}
    tmp_output.replace(output)

    deltas=np.concatenate(delta_parts) if delta_parts else np.array([],dtype=float)
    delta_median=float(np.quantile(deltas,0.5)) if len(deltas) else None
    delta_p95=float(np.quantile(deltas,0.95)) if len(deltas) else None
    delta_max=float(np.max(deltas)) if len(deltas) else None

    con=duckdb.connect()
    p=str(output).replace("'","''")
    duplicate_point_rows=int(con.execute(f"SELECT COALESCE(SUM(c-1),0) FROM (SELECT geom_lon,geom_lat,COUNT(*) c FROM read_parquet('{p}') GROUP BY 1,2 HAVING COUNT(*)>1)").fetchone()[0] or 0)
    duplicate_name_point_rows=int(con.execute(f"SELECT COALESCE(SUM(c-1),0) FROM (SELECT pname_raw,geom_lon,geom_lat,COUNT(*) c FROM read_parquet('{p}') GROUP BY 1,2,3 HAVING COUNT(*)>1)").fetchone()[0] or 0)
    unique_name_count=int(con.execute(f"SELECT COUNT(DISTINCT pname_raw) FROM read_parquet('{p}')").fetchone()[0] or 0)
    con.close()

    parquet_hash=file_sha256(output)
    result={
        'ok':True,'cached':False,'stage07f_handoff_ok':True,'normalization_fingerprint':payload['fingerprint'],
        'output_parquet_path':str(output),'parquet_exists':output.exists(),'parquet_sha256':parquet_hash,'parquet_size_bytes':output.stat().st_size,
        'schema_columns':columns,'source_gdb_path':str(gdb),'source_layer':payload['layer'],'source_feature_count_expected':6039158,
        'bbox_filter_rows':bbox_rows,'row_count':exact_rows,'invalid_geometry_count':0,'blank_name_count':blank_names,'unique_name_count':unique_name_count,
        'raw_xy_valid_count':raw_xy_valid,'raw_xy_within_1m_count':raw_xy_within_1m,'raw_xy_within_1m_share':raw_xy_within_1m/raw_xy_valid if raw_xy_valid else None,
        'raw_geom_delta_m_median':delta_median,'raw_geom_delta_m_p95':delta_p95,'raw_geom_delta_m_max':delta_max,
        'duplicate_point_rows':duplicate_point_rows,'duplicate_name_point_rows':duplicate_name_point_rows,
        'min_geom_lon':min_lon,'max_geom_lon':max_lon,'min_geom_lat':min_lat,'max_geom_lat':max_lat,'max_distance_to_beijing_km':max_distance,
        'coarse_bbox':bbox,'study_region':payload['region'],'gdal_version':run(['ogr2ogr','--version'],timeout=60)['stdout'].strip(),
    }
    result_path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    if staging.exists(): staging.unlink()
    data_volume.commit()
    return result

payload={
    'gdb_path':gdb_path,'layer':s07g.SOURCE_LAYER,'bbox':bbox,'region':{'latitude':region.latitude,'longitude':region.longitude,'radius_km':region.radius_km},
    'fingerprint':fingerprint,'chunksize':200000,
}
with app.run():
    worker_result=normalize_bcl_beijing.remote(payload)

print('worker ok:',worker_result.get('ok'))
print('worker error:',worker_result.get('error'))
print('cached:',worker_result.get('cached'))
print('bbox rows:',worker_result.get('bbox_filter_rows'))
print('exact 100km rows:',worker_result.get('row_count'))


## QC + freeze normalized artifact


In [ ]:
if not worker_result or not worker_result.get('ok'):
    raise RuntimeError(worker_result or {'error':'worker returned no result'})

decision=s07g.evaluate_normalization_qc(worker_result,region=region)
print('NORMALIZATION DECISION')
display(s07g.decision_frame(decision))
if decision.runner_status != 'ready_for_anchor_context_audit':
    raise RuntimeError(f'07g normalization blocked: {decision}')

summary_fields=[
    'bbox_filter_rows','row_count','blank_name_count','unique_name_count','raw_xy_valid_count','raw_xy_within_1m_count','raw_xy_within_1m_share',
    'raw_geom_delta_m_median','raw_geom_delta_m_p95','raw_geom_delta_m_max','duplicate_point_rows','duplicate_name_point_rows',
    'min_geom_lon','max_geom_lon','min_geom_lat','max_geom_lat','max_distance_to_beijing_km','parquet_size_bytes','parquet_sha256','gdal_version'
]
summary=pd.DataFrame([{key:worker_result.get(key) for key in summary_fields}])
display(summary.T.rename(columns={0:'value'}))

manifest=s07g.build_normalization_manifest(
    fingerprint=fingerprint,source_md5=source_md5,source_container_path=gdb_path,
    output_parquet_path=worker_result['output_parquet_path'],output_parquet_sha256=worker_result['parquet_sha256'],
    summary=worker_result,decision=decision,region=region,
)
(C/'normalization_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
summary.to_csv(C/'normalization_qc_summary.csv',index=False)
s07g.decision_frame(decision).to_csv(C/'normalization_decision.csv',index=False)
pd.DataFrame([{'name':region.name,'latitude':region.latitude,'longitude':region.longitude,'radius_km':region.radius_km,'bbox_min_lon':bbox[0],'bbox_min_lat':bbox[1],'bbox_max_lon':bbox[2],'bbox_max_lat':bbox[3]}]).to_csv(C/'study_region.csv',index=False)
print('saved:',C)
print('normalized parquet:',worker_result['output_parquet_path'])
print('runner_status:',decision.runner_status)


## Interpretation boundary

- The normalized subset is historical external context only.
- `PNAME` is preserved as raw source text; no lexical category is assigned here.
- geometry coordinates are authoritative for spatial work because Stage 07f verified the FileGDB CRS as EPSG:4326.
- raw `X/Y` agreement is descriptive QC, not a replacement CRS.
- only `runner_status == ready_for_anchor_context_audit` permits the later anchor/name-context stage.
